# 04 — Simulated Data Streaming for Live Occupancy Prediction

This notebook simulates a live sensor data stream and applies a trained Spark ML
pipeline to predict room occupancy in real time using **Spark Structured Streaming**.

**Workflow:**
1. Train a Logistic Regression pipeline on batch training data
2. Save the fitted model to disk
3. Initialize a Spark Structured Streaming query (CSV file source → memory sink)
4. Run a background thread that drip-feeds test rows as CSV files into the stream
5. Display a live prediction dashboard that refreshes as data arrives

Features used: `Temperature`, `Humidity`, `Light`, `CO2`, `HumidityRatio`  
Target: `Occupancy` (0 = empty, 1 = occupied)

## 1. Colab Setup

In [ ]:
from google.colab import drive, userdata
import sys

# Standard mount to access the config file initially
drive.mount('/content/drive')
PROJECT_ROOT = userdata.get('BIG_DATA_PATH')
sys.path.append(PROJECT_ROOT + "/src/")

from utils.config import initialize_project
initialize_project()

## 2. Imports

In [ ]:
import os
import sys
import time
import shutil
import threading
import pandas as pd
import numpy as np
from IPython.display import clear_output, display

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
from pyspark.sql.functions import col
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

## 3. Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("RoomOccupancyLiveStream")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} session ready.")

## 4. Configuration

In [ ]:
FEATURE_COLS = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
LABEL_COL    = "Occupancy"

DATA_DIR         = "data/raw"          # relative to project root (set by initialize_project)
MODEL_PATH       = "/tmp/occ_model"
STREAM_INPUT_DIR = "/tmp/occ_stream_input"
CHECKPOINT_DIR   = "/tmp/occ_checkpoint"

BATCH_SIZE     = 10    # sensor rows written per micro-batch
BATCH_DELAY    = 2.0   # seconds between batches (simulates sensor interval)
N_BATCHES      = 20    # total batches to emit
DISPLAY_ROUNDS = 30    # max iterations of the live dashboard loop

## 5. Load Data

In [ ]:
train_pdf = pd.read_csv(f"{DATA_DIR}/datatraining.txt", index_col=0)
test_pdf  = pd.read_csv(f"{DATA_DIR}/datatest2.txt",   index_col=0)

print(f"Training rows : {len(train_pdf)}")
print(f"Test rows     : {len(test_pdf)}")
print("\nSample (first 3 training rows):")
display(train_pdf[FEATURE_COLS + [LABEL_COL]].head(3))

## 6. Train & Save the Spark ML Pipeline

We train on batch data first, then save the fitted `PipelineModel` so the
streaming query can load it without needing the training data again.

In [ ]:
train_sdf = spark.createDataFrame(train_pdf[FEATURE_COLS + [LABEL_COL]])

assembler = VectorAssembler(
    inputCols=FEATURE_COLS, outputCol="raw_features"
)
scaler = StandardScaler(
    inputCol="raw_features", outputCol="features",
    withMean=True, withStd=True
)
lr = LogisticRegression(
    featuresCol="features", labelCol=LABEL_COL,
    maxIter=1000, regParam=0.01, elasticNetParam=0.0
)
pipeline = Pipeline(stages=[assembler, scaler, lr])

print("Training model on batch data...")
model = pipeline.fit(train_sdf)
print("Training complete.")

shutil.rmtree(MODEL_PATH, ignore_errors=True)
model.save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

## 7. Batch Accuracy Check

In [ ]:
test_sdf = spark.createDataFrame(test_pdf[FEATURE_COLS + [LABEL_COL]])
batch_preds = model.transform(test_sdf)

evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL, predictionCol="prediction", metricName="accuracy"
)
print(f"Batch test accuracy: {evaluator.evaluate(batch_preds):.4f}")

## 8. Set Up Streaming Infrastructure

We use a **file-based source**: a background thread writes small CSV batches
into `STREAM_INPUT_DIR`, and Spark Structured Streaming picks up each new
file as a micro-batch.

In [ ]:
for path in [STREAM_INPUT_DIR, CHECKPOINT_DIR]:
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path)
print("Streaming directories ready.")

### Define the Streaming Schema

Spark Structured Streaming requires an explicit schema for file sources.
We include `Occupancy` as a ground-truth label for accuracy tracking; the
model only sees the feature columns.

In [ ]:
stream_schema = StructType([
    StructField("Temperature",   DoubleType(),  True),
    StructField("Humidity",      DoubleType(),  True),
    StructField("Light",         DoubleType(),  True),
    StructField("CO2",           DoubleType(),  True),
    StructField("HumidityRatio", DoubleType(),  True),
    StructField("Occupancy",     IntegerType(), True),  # ground truth
])

### Start the Streaming Query

The query chain is:  
`readStream (CSV dir)` → `PipelineModel.transform()` → `writeStream (memory sink)`

Writing to the **memory sink** lets us query predictions interactively
with `spark.sql("SELECT * FROM live_predictions")` while the stream runs.

In [ ]:
stream_model = PipelineModel.load(MODEL_PATH)

stream_df = (
    spark.readStream
         .schema(stream_schema)
         .option("header", "true")
         .csv(STREAM_INPUT_DIR)
)

pred_stream = stream_model.transform(stream_df).select(
    "Temperature", "Humidity", "Light", "CO2", "HumidityRatio",
    col("Occupancy").alias("actual"),
    col("prediction").cast(IntegerType()).alias("predicted"),
)

query = (
    pred_stream.writeStream
               .queryName("live_predictions")
               .format("memory")
               .outputMode("append")
               .option("checkpointLocation", CHECKPOINT_DIR)
               .start()
)
print("Streaming query started — waiting for sensor data...")

## 9. Background Data Producer

This thread simulates a physical sensor by writing `BATCH_SIZE` rows every
`BATCH_DELAY` seconds as a new CSV file into the streaming input directory.

In [ ]:
stop_flag   = threading.Event()
stream_data = test_pdf[FEATURE_COLS + [LABEL_COL]].reset_index(drop=True)

def data_producer():
    for batch_idx in range(N_BATCHES):
        if stop_flag.is_set():
            break
        start = batch_idx * BATCH_SIZE
        batch = stream_data.iloc[start : start + BATCH_SIZE]
        if batch.empty:
            break
        out_file = os.path.join(STREAM_INPUT_DIR, f"batch_{batch_idx:04d}.csv")
        batch.to_csv(out_file, index=False)
        time.sleep(BATCH_DELAY)
    print("[Producer] Finished emitting sensor batches.")

producer_thread = threading.Thread(target=data_producer, daemon=True)
producer_thread.start()
print(
    f"Data producer started: "
    f"{N_BATCHES} batches x {BATCH_SIZE} rows every {BATCH_DELAY}s "
    f"({N_BATCHES * BATCH_SIZE} total rows)"
)

## 10. Live Prediction Dashboard

The cell below refreshes every 2 seconds, querying the in-memory sink
and displaying the latest predictions alongside a running accuracy metric.
It stops automatically once all batches have been processed.

In [ ]:
time.sleep(3)  # allow first batch to arrive before starting display

EXPECTED_ROWS = N_BATCHES * BATCH_SIZE

for round_num in range(1, DISPLAY_ROUNDS + 1):
    clear_output(wait=True)

    all_preds = spark.sql("SELECT * FROM live_predictions")
    total     = all_preds.count()

    print("=" * 65)
    print(f"  LIVE ROOM OCCUPANCY STREAM    records received: {total} / {EXPECTED_ROWS}")
    print("=" * 65)

    if total > 0:
        pdf = all_preds.toPandas()

        # Show the 10 most recently arrived rows
        latest = pdf.tail(10).copy().reset_index(drop=True)
        latest["Status"] = latest["predicted"].map(
            {0: "-- EMPTY --", 1: "** OCCUPIED **"}
        )
        latest["Match"] = (
            (latest["actual"] == latest["predicted"])
            .map({True: "correct", False: "WRONG"})
        )
        display(
            latest[
                ["Temperature", "Humidity", "Light", "CO2",
                 "HumidityRatio", "Status", "Match"]
            ]
        )

        accuracy   = (pdf["actual"] == pdf["predicted"]).mean()
        n_occupied = int((pdf["predicted"] == 1).sum())
        n_empty    = int((pdf["predicted"] == 0).sum())
        print(f"\nRunning accuracy : {accuracy:.4f}")
        print(
            f"Predicted breakdown : {n_occupied} OCCUPIED  |  {n_empty} EMPTY"
        )
    else:
        print("  Waiting for first sensor batch...")

    # Exit once all expected data is processed
    if not producer_thread.is_alive() and total >= EXPECTED_ROWS:
        print("\nAll batches processed — stopping dashboard.")
        break

    time.sleep(2)

## 11. Final Summary

In [ ]:
final_preds = spark.sql("SELECT * FROM live_predictions").toPandas()

if not final_preds.empty:
    final_acc  = (final_preds["actual"] == final_preds["predicted"]).mean()
    n_total    = len(final_preds)
    n_occupied = int((final_preds["predicted"] == 1).sum())
    n_empty    = int((final_preds["predicted"] == 0).sum())
    n_correct  = int((final_preds["actual"] == final_preds["predicted"]).sum())

    print("=" * 50)
    print("  STREAMING SIMULATION COMPLETE")
    print("=" * 50)
    print(f"  Total records streamed : {n_total}")
    print(f"  Correct predictions    : {n_correct}")
    print(f"  Final accuracy         : {final_acc:.4f}")
    print(f"  OCCUPIED predictions   : {n_occupied}")
    print(f"  EMPTY predictions      : {n_empty}")
    print("=" * 50)

## 12. Cleanup

In [ ]:
query.stop()
stop_flag.set()
print("Streaming query stopped.")

for path in [STREAM_INPUT_DIR, CHECKPOINT_DIR, MODEL_PATH]:
    shutil.rmtree(path, ignore_errors=True)
print("Temporary files removed.")